<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 컴퓨터 비전 · 04. U-Net

## U-Net: Convolutional Networks for Biomedical Image Segmentation

- **원 논문:** [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597)
- **저자 / 발표:** Olaf Ronneberger, Philipp Fischer, Thomas Brox · MICCAI 2015 (2015)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** contracting/expanding path와 skip concatenation으로 mask를 분할한다.

**축소하거나 생략한 부분:** 572×572 biomedical images, valid convolution, cropping, elastic deformation 대신 16×16 합성 mask와 same-padding 1-level U-Net을 사용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2 Fig. 1: contracting/expansive path | Task 1 MiniUNet | 입출력 해상도 |
| §2: cropped feature-map concatenation | Task 1 skip concat | decoder channels |
| §2.1 Eq. (1): pixel-wise softmax/CE | Task 2 BCE 이진 대응 | loss 감소 |
| §3 Fig. 3: overlap quality | Task 3 Dice | mask plot |

## 핵심 아이디어와 수식

$$\operatorname{Dice}(P,Y)=\frac{2\sum_i P_iY_i+\epsilon}{\sum_iP_i+\sum_iY_i+\epsilon}$$

**기호 해설:** P는 예측 mask, Y는 정답 mask, i는 pixel, ε은 0 나눗셈 방지 상수다.

**코드 대응:** Task 1은 encoder/decoder와 skip을 만들고 Task 2가 BCE, Task 3이 Dice를 구현한다.

**입력과 출력 shape:** 입력 [B,1,16,16] → bottleneck [B,12,8,8] → logits [B,1,16,16].

**포트폴리오 구현 범위:** 논문의 4-level 깊이와 crop은 줄였지만 localization skip과 pixel loss는 보존한다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2 Fig. 1: contracting/expansive path
- **노트북에서 구현할 부분:** Task 1 MiniUNet
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 입출력 해상도

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
@dataclass
class UNetConfig:
    encoder_channels: int = 6
    bottleneck_channels: int = 12
    learning_rate: float = 0.02
    steps: int = 16


def make_double_conv(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, 3, padding=1),
        nn.ReLU(),
    )


class MiniUNet(nn.Module):
    def __init__(self, config):
        super().__init__()
        c1 = config.encoder_channels
        c2 = config.bottleneck_channels
        self.encoder = make_double_conv(1, c1)
        self.bottleneck = make_double_conv(c1, c2)
        self.upsample = nn.ConvTranspose2d(c2, c1, 2, stride=2)
        self.decoder = make_double_conv(c1 * 2, c1)
        self.mask_head = nn.Conv2d(c1, 1, kernel_size=1)

    def encode(self, inputs):
        skip = self.encoder(inputs)
        low = self.bottleneck(F.max_pool2d(skip, 2))
        return skip, low

    def forward(self, inputs):
        skip, low = self.encode(inputs)
        upsampled = self.upsample(low)
        decoded = self.decoder(torch.cat((upsampled, skip), dim=1))
        return self.mask_head(decoded)


config = UNetConfig()
model = MiniUNet(config).to(DEVICE)
assert model(images[:2]).shape == (2, 1, 16, 16)
assert model.decoder[0].in_channels == 12

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2: cropped feature-map concatenation
- **노트북에서 구현할 부분:** Task 1 skip concat
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** decoder channels

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
def segmentation_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets)


def train_segmentation_step(model, optimizer, inputs, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs)
    loss = segmentation_loss(logits, targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


target_masks = masks[train_idx, None]
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    loss = train_segmentation_step(
        model,
        optimizer,
        x_train,
        target_masks,
    )
    losses.append(loss)
assert min(losses[3:]) < losses[0]

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3 Fig. 3: overlap quality
- **노트북에서 구현할 부분:** Task 3 Dice
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** mask plot

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
def dice_score(predictions, targets, epsilon=1.0):
    dimensions = (1, 2, 3)
    intersection = (predictions * targets).sum(dim=dimensions)
    total = predictions.sum(dim=dimensions) + targets.sum(dim=dimensions)
    return ((2 * intersection + epsilon) / (total + epsilon)).mean()


def evaluate_segmentation(model, inputs, targets):
    model.eval()
    with torch.no_grad():
        probabilities = model(inputs).sigmoid()
        predictions = (probabilities > 0.5).float()
    return float(dice_score(predictions, targets)), predictions


truth = masks[test_idx, None]
dice, predictions = evaluate_segmentation(model, images[test_idx], truth)
assert 0.0 <= dice <= 1.0
figure, axes = plt.subplots(1, 3, figsize=(8, 2.7))
axes[0].plot(losses)
axes[0].set_title("pixel BCE")
axes[1].imshow(truth[0, 0].detach().cpu())
axes[1].set_title("target")
axes[2].imshow(predictions[0, 0].detach().cpu())
axes[2].set_title(f"prediction Dice={dice:.2f}")
plt.tight_layout()
plt.show()

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 572×572 biomedical images, valid convolution, cropping, elastic deformation 대신 16×16 합성 mask와 same-padding 1-level U-Net을 사용한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.